# Component 3 — Leakage-Safe Model Training

This notebook compares transparent baselines and fixed regression candidates using expanding-window cross-validation, then evaluates candidate selection on the 2023 validation period. The 2024 final-test outcomes remain untouched.

## Training safeguards

- Target: `target_log1p_median_catch`; kilogram predictions are recovered with `expm1` and clipped at zero.
- Preprocessing is contained inside scikit-learn pipelines and fitted separately in every fold.
- Numeric values use training-median imputation with missingness indicators.
- Categories use training-most-frequent imputation and one-hot encoding with unknown-category tolerance.
- Fixed model configurations avoid tuning repeatedly against the 2023 validation year.
- Primary selection metric: log-scale MAE; kilogram MAE/RMSE and bias remain important interpretability diagnostics.
- 2024 rows are counted only to verify their exclusion; their predictors and targets are never passed to fitting, prediction or scoring.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4','sklearn':'scikit-learn>=1.3,<2','joblib':'joblib>=1.3,<2'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Model-training dependencies are available.')

In [ ]:
from __future__ import annotations
import hashlib, json, os, sys, time
from pathlib import Path
import joblib
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
try:
    from IPython.display import display
except ImportError:
    display=print
pd.set_option('display.max_columns',120); plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE=42
COLORS={'baseline':'#bab0ac','ridge':'#4e79a7','random_forest':'#59a14f','extra_trees':'#8f63b8','hist_gradient_boosting':'#f28e2b'}

## 1. Locate inputs and freeze feature roles

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
PROCESSED_DIR=Path(os.getenv('FISHGUARD_C3_PROCESSED_DIR',str(ROOT/'data'/'processed'))).resolve()
MODEL_DIR=Path(os.getenv('FISHGUARD_C3_MODEL_DIR',str(ROOT/'models'))).resolve()
EVIDENCE_DIR=Path(os.getenv('FISHGUARD_C3_TRAIN_AUDIT',str(ROOT/'outputs'/'audit'/'model_training'))).resolve()
METRIC_DIR=Path(os.getenv('FISHGUARD_C3_TRAIN_METRICS',str(ROOT/'outputs'/'metrics'/'model_training'))).resolve()
PREDICTION_DIR=Path(os.getenv('FISHGUARD_C3_TRAIN_PREDICTIONS',str(ROOT/'outputs'/'predictions'/'model_training'))).resolve()
FIGURE_DIR=Path(os.getenv('FISHGUARD_C3_TRAIN_FIGURES',str(ROOT/'outputs'/'figures'/'model_training'))).resolve()
for directory in [MODEL_DIR,EVIDENCE_DIR,METRIC_DIR,PREDICTION_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
DATA_PATH=PROCESSED_DIR/'monthly_model_features_split.parquet'
DICTIONARY_PATH=ROOT/'outputs'/'audit'/'feature_engineering'/'feature_dictionary.csv'
FOLDS_PATH=ROOT/'outputs'/'audit'/'modelling_eda_split'/'expanding_window_cv_folds.csv'
if not DATA_PATH.is_file(): raise FileNotFoundError(f'Missing {DATA_PATH}. Run 07_modelling_eda_and_split.ipynb first.')
for path in [DICTIONARY_PATH,FOLDS_PATH]:
    if not path.is_file(): raise FileNotFoundError(f'Missing {path}.')
data=pd.read_parquet(DATA_PATH); dictionary=pd.read_csv(DICTIONARY_PATH); fold_definitions=pd.read_csv(FOLDS_PATH)
data['month_start']=pd.to_datetime(data.month_start,errors='raise')
predictors=dictionary.loc[dictionary.role=='candidate_predictor','column'].tolist()
missing_predictors=sorted(set(predictors)-set(data.columns))
if missing_predictors: raise KeyError(f'Missing predictors: {missing_predictors}')
TARGET='target_log1p_median_catch'
train=data.loc[data.data_split.astype(str)=='train'].copy(); validation=data.loc[data.data_split.astype(str)=='validation'].copy(); test_row_count=int((data.data_split.astype(str)=='test').sum())
numeric_features=[column for column in predictors if pd.api.types.is_numeric_dtype(train[column])]
categorical_features=[column for column in predictors if column not in numeric_features]
source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('Train:',len(train),'Validation:',len(validation),'Sealed test rows:',test_row_count)
print('Predictors:',len(predictors),'Numeric:',len(numeric_features),'Categorical:',categorical_features)

## 2. Define metrics, baselines and leakage-safe pipelines

In [ ]:
def regression_metrics(y_true_log,y_pred_log):
    y_true_log=np.asarray(y_true_log,dtype=float); y_pred_log=np.asarray(y_pred_log,dtype=float)
    y_true_kg=np.clip(np.expm1(y_true_log),a_min=0,a_max=None); y_pred_kg=np.clip(np.expm1(np.clip(y_pred_log,a_min=0,a_max=None)),a_min=0,a_max=None)
    return {'log_mae':mean_absolute_error(y_true_log,y_pred_log),'log_rmse':mean_squared_error(y_true_log,y_pred_log)**0.5,'log_median_ae':median_absolute_error(y_true_log,y_pred_log),'log_r2':r2_score(y_true_log,y_pred_log),'kg_mae':mean_absolute_error(y_true_kg,y_pred_kg),'kg_rmse':mean_squared_error(y_true_kg,y_pred_kg)**0.5,'kg_median_ae':median_absolute_error(y_true_kg,y_pred_kg),'kg_bias':float(np.mean(y_pred_kg-y_true_kg))}
def historical_baseline(frame,fallback_log):
    values=frame['group_median_catch_weight_kg_lag_1'].combine_first(frame['species_median_catch_weight_kg_lag_1']).combine_first(frame['cell_median_catch_weight_kg_lag_1']).combine_first(frame['global_median_catch_weight_kg_lag_1'])
    return np.log1p(values.clip(lower=0)).fillna(fallback_log).to_numpy()
numeric_scaled=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),('scaler',StandardScaler())])
numeric_tree=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True))])
categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
linear_preprocessor=ColumnTransformer([('numeric',numeric_scaled,numeric_features),('categorical',categorical,categorical_features)],verbose_feature_names_out=False)
tree_preprocessor=ColumnTransformer([('numeric',numeric_tree,numeric_features),('categorical',categorical,categorical_features)],verbose_feature_names_out=False)
models={
 'ridge':Pipeline([('preprocess',linear_preprocessor),('model',Ridge(alpha=10.0,solver='lsqr'))]),
 'random_forest':Pipeline([('preprocess',tree_preprocessor),('model',RandomForestRegressor(n_estimators=250,max_depth=None,min_samples_leaf=5,max_features=.7,n_jobs=-1,random_state=RANDOM_STATE))]),
 'extra_trees':Pipeline([('preprocess',tree_preprocessor),('model',ExtraTreesRegressor(n_estimators=250,max_depth=None,min_samples_leaf=5,max_features=.8,n_jobs=-1,random_state=RANDOM_STATE))]),
 'hist_gradient_boosting':Pipeline([('preprocess',tree_preprocessor),('model',HistGradientBoostingRegressor(max_iter=250,learning_rate=.05,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=1.0,random_state=RANDOM_STATE))])}
model_configuration=pd.DataFrame([
 {'model':'historical_baseline','family':'baseline','configuration':'group lag1 → species lag1 → cell lag1 → global lag1 → training median'},
 {'model':'median_baseline','family':'baseline','configuration':'training-target median'},
 {'model':'ridge','family':'linear','configuration':'Ridge alpha=10; scaled numeric; one-hot categorical'},
 {'model':'random_forest','family':'bagged trees','configuration':'250 trees; min leaf=5; max_features=0.7'},
 {'model':'extra_trees','family':'randomized trees','configuration':'250 trees; min leaf=5; max_features=0.8'},
 {'model':'hist_gradient_boosting','family':'boosted trees','configuration':'250 iterations; rate=0.05; max leaves=31; min leaf=20; L2=1'}])
model_configuration.to_csv(EVIDENCE_DIR/'model_configurations.csv',index=False)
display(model_configuration)

## 3. Expanding-window cross-validation

In [ ]:
cv_rows=[]
for fold in fold_definitions.itertuples(index=False):
    fold_train=train.loc[train.month_start<=pd.Timestamp(fold.train_end)]; fold_valid=train.loc[(train.month_start>=pd.Timestamp(fold.validation_start))&(train.month_start<=pd.Timestamp(fold.validation_end))]
    X_train=fold_train[predictors]; y_train=fold_train[TARGET]; X_valid=fold_valid[predictors]; y_valid=fold_valid[TARGET]
    fallback=float(y_train.median())
    baseline_predictions={'median_baseline':np.full(len(y_valid),fallback),'historical_baseline':historical_baseline(fold_valid,fallback)}
    for name,predictions in baseline_predictions.items():
        cv_rows.append({'fold_id':int(fold.fold_id),'model':name,'train_rows':len(fold_train),'validation_rows':len(fold_valid),'fit_seconds':0.0,**regression_metrics(y_valid,predictions)})
    for name,pipeline in models.items():
        fitted=clone(pipeline); started=time.perf_counter(); fitted.fit(X_train,y_train); fit_seconds=time.perf_counter()-started; predictions=fitted.predict(X_valid)
        cv_rows.append({'fold_id':int(fold.fold_id),'model':name,'train_rows':len(fold_train),'validation_rows':len(fold_valid),'fit_seconds':fit_seconds,**regression_metrics(y_valid,predictions)})
    print(f'Completed fold {int(fold.fold_id)}: {len(fold_train):,} → {len(fold_valid):,} rows')
cv_metrics=pd.DataFrame(cv_rows)
metric_columns=['log_mae','log_rmse','log_median_ae','log_r2','kg_mae','kg_rmse','kg_median_ae','kg_bias','fit_seconds']
cv_summary=cv_metrics.groupby('model')[metric_columns].agg(['mean','std']).reset_index()
cv_summary.columns=['model']+[f'{metric}_{stat}' for metric,stat in cv_summary.columns.tolist()[1:]]
cv_summary=cv_summary.sort_values('log_mae_mean')
cv_metrics.to_csv(METRIC_DIR/'expanding_window_cv_metrics.csv',index=False); cv_summary.to_csv(METRIC_DIR/'expanding_window_cv_summary.csv',index=False)
display(cv_summary[['model','log_mae_mean','log_mae_std','kg_mae_mean','kg_rmse_mean','log_r2_mean','fit_seconds_mean']])

## 4. Fit on all training data and evaluate 2023 validation

In [ ]:
X_train=train[predictors]; y_train=train[TARGET]; X_validation=validation[predictors]; y_validation=validation[TARGET]
validation_rows=[]; validation_predictions=[]; fitted_models={}
fallback=float(y_train.median())
baseline_predictions={'median_baseline':np.full(len(validation),fallback),'historical_baseline':historical_baseline(validation,fallback)}
for name,predictions in baseline_predictions.items():
    validation_rows.append({'model':name,'fit_seconds':0.0,**regression_metrics(y_validation,predictions)})
    for index,prediction in zip(validation.index,predictions): validation_predictions.append({'row_index':int(index),'model':name,'prediction_log':prediction})
for name,pipeline in models.items():
    fitted=clone(pipeline); started=time.perf_counter(); fitted.fit(X_train,y_train); fit_seconds=time.perf_counter()-started; predictions=fitted.predict(X_validation); fitted_models[name]=fitted
    validation_rows.append({'model':name,'fit_seconds':fit_seconds,**regression_metrics(y_validation,predictions)})
    for index,prediction in zip(validation.index,predictions): validation_predictions.append({'row_index':int(index),'model':name,'prediction_log':prediction})
validation_metrics=pd.DataFrame(validation_rows).sort_values('log_mae')
prediction_long=pd.DataFrame(validation_predictions)
prediction_keys=validation[['month_start','spatial_cell_id','species_model','gear_model',TARGET,'target_median_catch_weight_kg']].reset_index().rename(columns={'index':'row_index','target_median_catch_weight_kg':'actual_kg'})
prediction_long=prediction_long.merge(prediction_keys,on='row_index',how='left',validate='m:1')
prediction_long['prediction_kg']=np.expm1(prediction_long.prediction_log.clip(lower=0))
validation_metrics.to_csv(METRIC_DIR/'validation_metrics.csv',index=False); prediction_long.to_csv(PREDICTION_DIR/'validation_predictions.csv',index=False)
display(validation_metrics)

## 5. Select and save the development-stage candidate

In [ ]:
cv_rank=cv_summary[['model','log_mae_mean','log_mae_std']].copy(); cv_rank['cv_rank']=cv_rank.log_mae_mean.rank(method='min')
ranking=validation_metrics.merge(cv_rank,on='model',how='left',validate='1:1'); ranking['validation_rank']=ranking.log_mae.rank(method='min'); ranking['combined_rank']=ranking.cv_rank+ranking.validation_rank
ranking=ranking.sort_values(['combined_rank','log_mae','log_mae_mean']).reset_index(drop=True)
learned_ranking=ranking.loc[ranking.model.isin(models)].copy(); selected_name=str(learned_ranking.iloc[0].model); selected_pipeline=fitted_models[selected_name]
best_baseline=float(ranking.loc[ranking.model.isin(['median_baseline','historical_baseline']),'log_mae'].min()); selected_validation_mae=float(learned_ranking.iloc[0].log_mae)
candidate_beats_baseline=selected_validation_mae<best_baseline
MODEL_PATH=MODEL_DIR/'selected_development_pipeline.joblib'; joblib.dump(selected_pipeline,MODEL_PATH)
model_sha256=hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
ranking.to_csv(METRIC_DIR/'model_ranking.csv',index=False)
selection=pd.DataFrame([{'selected_model':selected_name,'selection_rule':'lowest combined expanding-window CV rank plus 2023 validation rank among learned models','validation_log_mae':selected_validation_mae,'best_baseline_validation_log_mae':best_baseline,'candidate_beats_baseline':candidate_beats_baseline,'model_file':MODEL_PATH.name,'model_sha256':model_sha256}])
selection.to_csv(EVIDENCE_DIR/'model_selection.csv',index=False)
display(ranking[['model','log_mae_mean','log_mae_std','log_mae','kg_mae','kg_rmse','log_r2','combined_rank']]); display(selection)

## 6. Validation diagnostics and permutation importance

In [ ]:
selected_predictions=prediction_long.loc[prediction_long.model==selected_name].copy()
selected_predictions['residual_log']=selected_predictions[TARGET]-selected_predictions.prediction_log
importance=permutation_importance(selected_pipeline,X_validation,y_validation,scoring='neg_mean_absolute_error',n_repeats=5,random_state=RANDOM_STATE,n_jobs=-1)
importance_table=pd.DataFrame({'feature':predictors,'importance_mean':importance.importances_mean,'importance_std':importance.importances_std}).sort_values('importance_mean',ascending=False)
importance_table.to_csv(METRIC_DIR/'validation_permutation_importance.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,6))
axes[0].scatter(selected_predictions[TARGET],selected_predictions.prediction_log,alpha=.5,color=COLORS.get(selected_name,'#4e79a7')); limits=[min(selected_predictions[TARGET].min(),selected_predictions.prediction_log.min()),max(selected_predictions[TARGET].max(),selected_predictions.prediction_log.max())]; axes[0].plot(limits,limits,'--',color='black'); axes[0].set_xlim(limits); axes[0].set_ylim(limits); axes[0].set_xlabel('Actual log1p catch'); axes[0].set_ylabel('Predicted log1p catch'); axes[0].set_title(f'{selected_name}: validation predictions')
axes[1].scatter(selected_predictions.prediction_log,selected_predictions.residual_log,alpha=.5,color=COLORS.get(selected_name,'#4e79a7')); axes[1].axhline(0,color='black',linestyle='--'); axes[1].set_xlabel('Predicted log1p catch'); axes[1].set_ylabel('Actual − predicted'); axes[1].set_title('Validation residuals')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'selected_model_validation_diagnostics.png',dpi=200,bbox_inches='tight'); plt.show()
top=importance_table.head(20).sort_values('importance_mean'); fig,ax=plt.subplots(figsize=(11,8)); ax.barh(top.feature,top.importance_mean,xerr=top.importance_std,color=COLORS.get(selected_name,'#4e79a7')); ax.axvline(0,color='black',linewidth=.8); ax.set_xlabel('Increase in validation MAE when permuted'); ax.set_title(f'{selected_name}: validation permutation importance'); fig.tight_layout(); fig.savefig(FIGURE_DIR/'validation_permutation_importance.png',dpi=200,bbox_inches='tight'); plt.show(); display(importance_table.head(20))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,6))
cv_plot=cv_summary.sort_values('log_mae_mean'); axes[0].barh(cv_plot.model,cv_plot.log_mae_mean,xerr=cv_plot.log_mae_std,color=[COLORS.get(name,COLORS['baseline']) for name in cv_plot.model]); axes[0].set_xlabel('Mean log MAE ± fold SD'); axes[0].set_title('Expanding-window cross-validation')
val_plot=validation_metrics.sort_values('log_mae'); axes[1].barh(val_plot.model,val_plot.log_mae,color=[COLORS.get(name,COLORS['baseline']) for name in val_plot.model]); axes[1].set_xlabel('2023 validation log MAE'); axes[1].set_title('Held-out validation comparison')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'model_comparison.png',dpi=200,bbox_inches='tight'); plt.show()

## 7. Save training checks, findings and manifest

In [ ]:
training_checks=pd.DataFrame([
 {'check':'test excluded from fitting and scoring','passed':test_row_count>0 and len(train)+len(validation)+test_row_count==len(data),'detail':f'{test_row_count} sealed rows'},
 {'check':'target excluded from predictors','passed':TARGET not in predictors and not any(column.startswith('target_') for column in predictors),'detail':f'{len(predictors)} predictors'},
 {'check':'diagnostics excluded from predictors','passed':not any(column.startswith('diagnostic_') for column in predictors),'detail':'current-month totals/counts excluded'},
 {'check':'all preprocessing inside pipelines','passed':all(isinstance(model,Pipeline) for model in models.values()),'detail':'imputation, scaling and encoding refit per fold'},
 {'check':'all configured models evaluated','passed':set(model_configuration.model)==set(cv_metrics.model)==set(validation_metrics.model),'detail':f'{len(model_configuration)} models including baselines'},
 {'check':'predictions finite','passed':np.isfinite(prediction_long.prediction_log).all(),'detail':f'{len(prediction_long):,} validation predictions'},
 {'check':'saved pipeline hash recorded','passed':MODEL_PATH.is_file() and hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()==model_sha256,'detail':MODEL_PATH.name}])
assert training_checks.passed.all(),training_checks.loc[~training_checks.passed].to_dict('records')
training_checks.to_csv(EVIDENCE_DIR/'training_leakage_checks.csv',index=False)
selected_metrics=validation_metrics.set_index('model').loc[selected_name]
findings=pd.DataFrame([
 {'finding':'Selected development candidate','result':selected_name},
 {'finding':'Selection evidence','result':'Three expanding-window folds plus held-out 2023 validation; 2024 not scored'},
 {'finding':'Validation log MAE','result':f"{selected_metrics.log_mae:.4f}"},
 {'finding':'Validation kilogram MAE','result':f"{selected_metrics.kg_mae:.2f} kg"},
 {'finding':'Validation kilogram bias','result':f"{selected_metrics.kg_bias:.2f} kg (prediction minus actual)"},
 {'finding':'Baseline comparison','result':f"Selected learned candidate beats best validation baseline: {candidate_beats_baseline}"},
 {'finding':'Interpretation boundary','result':'Permutation importance is validation-specific association, not causality'},
 {'finding':'Next stage','result':'Refit the frozen selected configuration on train+validation, evaluate 2024 exactly once, then save the final deployment artifact'}])
findings.to_csv(EVIDENCE_DIR/'model_training_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','input':DATA_PATH.name,'input_sha256':source_sha256,'target':TARGET,'train_rows':len(train),'validation_rows':len(validation),'sealed_test_rows':test_row_count,'test_target_used':False,'predictors':predictors,'numeric_features':len(numeric_features),'categorical_features':categorical_features,'random_state':RANDOM_STATE,'models_compared':model_configuration.model.tolist(),'selection_rule':'minimum combined CV and validation rank among learned models','selected_model':selected_name,'candidate_beats_baseline':bool(candidate_beats_baseline),'saved_model':MODEL_PATH.name,'saved_model_sha256':model_sha256,'final_model_refit':False,'final_test_evaluated':False,'figures':sorted(p.name for p in FIGURE_DIR.glob('*.png')),'audit':sorted(p.name for p in EVIDENCE_DIR.glob('*.csv')),'metrics':sorted(p.name for p in METRIC_DIR.glob('*.csv')),'predictions':sorted(p.name for p in PREDICTION_DIR.glob('*.csv'))}
with (EVIDENCE_DIR/'model_training_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(training_checks); display(findings); print('Selected pipeline:',MODEL_PATH)

## Viva interpretation

The selected development candidate is not yet the final reported model. It earned selection from past-to-future folds and the 2023 validation period while the 2024 outcomes remained untouched. Performance must be compared with both a constant median baseline and a historical-lag baseline. Only after this notebook is accepted should the selected configuration be refitted on training plus validation and evaluated once on 2024.